# Reporte Consolidado — Predicción de Hospitalización por Dengue
## Bucaramanga 2015–2025 · 28,626 casos · SIVIGILA

---

## Objetivo del notebook

Sintetizar los hallazgos de los cuatro módulos analíticos en una
narrativa coherente para la presentación de maestría.
No se recalcula nada — todo se carga desde los checkpoints y tablas
generadas en los módulos anteriores.

| Módulo | Notebook | Hallazgo principal |
|--------|----------|-------------------|
| 02 | `02a/02b` — Análisis clínico | OR ajustados, VIF, modelos explicativos |
| 03a/03c | Modelado supervisado | DCSA F1=0.800 · DSSA AUC-PR=0.096 |
| 03b/03d | Validación temporal | Ruptura estructural DCSA · drift epidemiológico DSSA |
| 04 | Clustering | Estructura dominada por sesgo de registro (DCSA) |

**Estructura del reporte:**
1. Resumen ejecutivo
2. Comparativa DCSA vs DSSA
3. Rendimiento de los modelos supervisados
4. Validación temporal — robustez
5. Hallazgos del clustering
6. Implicaciones clínicas
7. Limitaciones y trabajo futuro

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import warnings
import joblib
from pathlib import Path

warnings.filterwarnings('ignore')
%matplotlib inline

plt.rcParams.update({
    'figure.dpi': 150, 'savefig.dpi': 300,
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.titlesize': 12, 'axes.titleweight': 'bold',
    'axes.labelsize': 10, 'legend.frameon': False,
})
COLORS = {
    'primary':    '#2C7BB6',
    'secondary':  '#D7191C',
    'accent':     '#FDAE61',
    'neutral':    '#7F7F7F',
    'con_signos': '#E74C3C',
    'sin_signos': '#3498DB',
    'verde':      '#2ECC71',
}

ROOT        = Path(r'D:\Documentos\GitHub\dengue')
OUT_FIG     = ROOT / 'outputs' / 'figures'
OUT_TAB     = ROOT / 'outputs' / 'tables'
CHECKPOINTS = ROOT / 'outputs' / 'checkpoints'

# Cargar resultados de todos los módulos
mejor_dcsa   = joblib.load(CHECKPOINTS / 'mejor_modelo_DCSA.pkl')
mejor_dssa   = joblib.load(CHECKPOINTS / 'mejor_modelo_DSSA.pkl')
eval_dcsa    = joblib.load(CHECKPOINTS / 'eval_test_DCSA.pkl')
eval_dssa    = joblib.load(CHECKPOINTS / 'eval_test_DSSA.pkl')
imp_dcsa     = joblib.load(CHECKPOINTS / 'importancia_DCSA.pkl')
imp_dssa     = joblib.load(CHECKPOINTS / 'importancia_DSSA.pkl')

tabla_dcsa = eval_dcsa['tabla_test']
tabla_dssa = eval_dssa['tabla_test']
nombre_dcsa = eval_dcsa['mejor']
nombre_dssa = eval_dssa['mejor']

print('Checkpoints cargados:')
print(f'  Mejor modelo DCSA : {nombre_dcsa}')
print(f'  Mejor modelo DSSA : {nombre_dssa}')
print(f'  Tabla test DCSA   : {len(tabla_dcsa)} modelos')
print(f'  Tabla test DSSA   : {len(tabla_dssa)} modelos')

## 1. Resumen ejecutivo

Este estudio desarrolló modelos de clasificación supervisada para
predecir hospitalización por dengue en Bucaramanga (2015–2025)
a partir de datos del SIVIGILA, estratificando por clasificación clínica OMS.

Los resultados revelan tres hallazgos principales:

**1. El techo informativo del SIVIGILA:**
Los modelos alcanzan el límite de lo que los features disponibles permiten predecir.
En DCSA (F1=0.800) y DSSA (AUC-PR=0.096) la señal clínica en el formulario
de vigilancia epidemiológica es insuficiente para discriminación robusta.
Más datos o algoritmos más complejos no mejorarían el rendimiento —
se requieren features adicionales (laboratorio, signos vitales cuantitativos).

**2. La ruptura estructural 2021–2022 en DCSA:**
El cambio de protocolo INS en 2021 hizo incompatibles los datos de ambos períodos.
El modelo entrenado con datos mixtos (03a) no generaliza al período 2022–2025
(Balanced Accuracy = 0.564 ≈ azar). El problema no es técnico sino de
calidad y consistencia del sistema de vigilancia epidemiológica.

**3. El clustering captura sesgo de registro, no perfiles clínicos:**
En DCSA, K-Means identifica un cluster con todos los síntomas en cero
(formularios incompletos pre-2022) que no representa un perfil clínico real.
La estructura encontrada por métodos distintos diverge completamente
(ARI K-Means vs K-Prototypes = 0.004), señal de que la estructura
es débil e inestable.

In [ ]:
# Tabla resumen ejecutivo — una fila por estrato y modelo
resumen = pd.DataFrame([
    {
        'Estrato': 'DCSA (con signos)',
        'N': '12,713',
        'Hosp.': '64.8%',
        'Mejor modelo': nombre_dcsa,
        'Métrica primaria': 'F1-score',
        'Valor test': tabla_dcsa.loc[nombre_dcsa, 'F1'] if nombre_dcsa in tabla_dcsa.index else tabla_dcsa.iloc[0]['F1'],
        'AUC-PR test': tabla_dcsa.loc[nombre_dcsa, 'AUC-PR'] if 'AUC-PR' in tabla_dcsa.columns else tabla_dcsa.iloc[0].get('AUC_PR','—'),
        'Brier Score': tabla_dcsa.loc[nombre_dcsa, 'Brier Score'] if 'Brier Score' in tabla_dcsa.columns else '—',
        'Bal. Acc.': tabla_dcsa.loc[nombre_dcsa, 'Bal. Acc'] if 'Bal. Acc' in tabla_dcsa.columns else tabla_dcsa.iloc[0].get('Bal_Acc','—'),
    },
    {
        'Estrato': 'DSSA (sin signos)',
        'N': '15,734',
        'Hosp.': '3.9%',
        'Mejor modelo': nombre_dssa,
        'Métrica primaria': 'AUC-PR',
        'Valor test': tabla_dssa.loc[nombre_dssa, 'AUC-PR'] if nombre_dssa in tabla_dssa.index and 'AUC-PR' in tabla_dssa.columns else tabla_dssa.iloc[0].get('AUC-PR','—'),
        'AUC-PR test': tabla_dssa.loc[nombre_dssa, 'AUC-PR'] if nombre_dssa in tabla_dssa.index and 'AUC-PR' in tabla_dssa.columns else '—',
        'Brier Score': tabla_dssa.loc[nombre_dssa, 'Brier Score'] if nombre_dssa in tabla_dssa.index and 'Brier Score' in tabla_dssa.columns else '—',
        'Bal. Acc.': tabla_dssa.loc[nombre_dssa, 'Bal. Acc'] if nombre_dssa in tabla_dssa.index and 'Bal. Acc' in tabla_dssa.columns else '—',
    }
])
display(resumen.set_index('Estrato'))
resumen.to_csv(OUT_TAB / 'reporte_resumen_ejecutivo.csv', index=False)

## 2. Comparativa DCSA vs DSSA

Los dos estratos representan problemas fundamentalmente diferentes
que requieren estrategias metodológicas distintas:

| Dimensión | DCSA | DSSA |
|-----------|------|------|
| Prevalencia hospitalización | 64.8% | 3.9% |
| Desbalance | Moderado (65/35) | Severo (96/4) |
| Señal clínica | Fuerte | Débil |
| Métrica primaria | F1-score | AUC-PR |
| SMOTE k_neighbors | 5 | 3 |
| Techo informativo alcanzado | Sí (~3,000 casos) | Sí (señal débil) |
| Ruptura temporal | Sí — masiva (+47pp hosp.) | No (~+0.3pp) |
| Drift tipo | Metodológico (protocolo INS) | Epidemiológico (serotipo) |
| Predictor principal | Síntomas generales | Estacionalidad (SEMANA) |
| Signos de alarma OMS | Importancia moderada | Importancia ≈ 0 |
| Variables demográficas | Baja importancia | Moderada (alerta equidad) |
| Umbral recomendado | Screening-First viable | Capacity-First preferible |
| DCA 'Treat All' | Competidor fuerte | Ineficiente |

In [ ]:
# Gráfico comparativo de métricas test — DCSA vs DSSA
# Usa columnas disponibles en cada tabla
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Panel A: DCSA — top 3 modelos
metricas_plot_dcsa = [c for c in ['F1','AUC-ROC','AUC-PR','Bal. Acc','Recall','Specificity']
    if c in tabla_dcsa.columns]
x = np.arange(len(metricas_plot_dcsa))
w = 0.25
colores_m = [COLORS['con_signos'], COLORS['primary'], COLORS['accent']]
for i, (modelo, row) in enumerate(tabla_dcsa.iterrows()):
    vals = [row[m] for m in metricas_plot_dcsa]
    axes[0].bar(x + (i-1)*w, vals, w,
        color=colores_m[i % len(colores_m)], alpha=0.85,
        label=modelo, edgecolor='white')
axes[0].set_xticks(x)
axes[0].set_xticklabels(metricas_plot_dcsa, fontsize=9)
axes[0].set_ylim(0, 1.05)
axes[0].set_title('A. DCSA — Top 3 modelos en test')
axes[0].legend(fontsize=8)
axes[0].grid(axis='y', alpha=0.3)

# Panel B: DSSA — top 3 modelos
metricas_plot_dssa = [c for c in ['AUC-PR','AUC-ROC','F1','Bal. Acc','Recall','Specificity']
    if c in tabla_dssa.columns]
x2 = np.arange(len(metricas_plot_dssa))
colores_m2 = [COLORS['sin_signos'], COLORS['primary'], COLORS['accent']]
for i, (modelo, row) in enumerate(tabla_dssa.iterrows()):
    vals = [row[m] for m in metricas_plot_dssa]
    axes[1].bar(x2 + (i-1)*w, vals, w,
        color=colores_m2[i % len(colores_m2)], alpha=0.85,
        label=modelo, edgecolor='white')
axes[1].set_xticks(x2)
axes[1].set_xticklabels(metricas_plot_dssa, fontsize=9)
axes[1].set_ylim(0, 1.05)
axes[1].set_title('B. DSSA — Top 3 modelos en test')
axes[1].legend(fontsize=8)
axes[1].grid(axis='y', alpha=0.3)

plt.suptitle('Métricas en conjunto de prueba — DCSA vs DSSA',
    fontsize=13, fontweight='bold')
plt.tight_layout()
fig.savefig(OUT_FIG / 'reporte_comparativa_modelos.png',
    dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

## 3. Rendimiento de los modelos supervisados

### DCSA — Dengue con signos de alarma

**Extra Trees** lidera con F1=0.800 en test, con un perfil clínico claro:
Recall=0.775 (identifica el 77.5% de los hospitalizados reales) y
Specificity=0.699 (identifica el 69.9% de los ambulatorios).
El Brier Score de 0.182 indica calibración aceptable, aunque CatBoost
está mejor calibrado (0.176) — preferible para triaje por probabilidades.

**Convergencia del portafolio:** la diferencia F1 entre el mejor y el peor
modelo avanzado en CV es de 0.014 puntos — dentro de la variabilidad estadística.
Las curvas de aprendizaje confirman el techo informativo desde ~3,000 casos.

**Puntos de operación:**
- Screening-First (umbral=0.243): Recall=95.2%, alerta al 89% de la cohorte
- Capacity-First Top 5% (umbral=0.846): PPV=97.7%, enriquecimiento 1.51x
- DCA: utilidad clínica para umbrales pt = 0.40–0.70

### DSSA — Dengue sin signos de alarma

**Extra Trees** lidera en AUC-PR en test, pero CatBoost y LightGBM están
mejor calibrados (Brier más bajo). La divergencia entre métrica primaria
y calibración no tiene solución única — depende del objetivo clínico.

**Señal débil:** el AUC-PR de todos los modelos es bajo en términos absolutos.
El formulario SIVIGILA no captura las variables que realmente predicen
hospitalización en este estrato (laboratorio, signos vitales, comorbilidades).

**Umbral default inapropiado:** con 3.9% de prevalencia, CatBoost y LightGBM
casi nunca superan P=0.5 — Recall ≈ 0% con umbral default.
El Capacity-First (Top 5–10%) es el punto de operación más viable operacionalmente.

In [ ]:
# Importancia de variables comparativa DCSA vs DSSA
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

color_tipo = {
    'Signo de alarma': COLORS['con_signos'],
    'Sintoma general': COLORS['sin_signos'],
    'Demografica':     COLORS['accent'],
    'Estacionalidad':  COLORS['neutral'],
}

for ax, df_imp, titulo, metrica in zip(
    axes,
    [imp_dcsa, imp_dssa],
    ['DCSA — Extra Trees (scoring: F1)', 'DSSA — Extra Trees (scoring: AUC-PR)'],
    ['F1', 'AUC-PR']
):
    df_imp = df_imp[df_imp['Variable'] != 'AÑO'].copy()
    df_top = df_imp.head(12)
    colores_b = [color_tipo.get(t, COLORS['neutral']) for t in df_top['Tipo']]
    bars = ax.barh(
        df_top['Variable'][::-1], df_top['Importancia'][::-1],
        xerr=df_top['Std'][::-1], color=colores_b[::-1],
        capsize=3, alpha=0.87, edgecolor='white', height=0.6
    )
    for bar, (_, row) in zip(bars, df_top[::-1].iterrows()):
        if row['Importancia'] > 0:
            ax.text(bar.get_width() + row['Std'] + 0.0003,
                bar.get_y() + bar.get_height()/2,
                f"{row['Importancia']:.4f}",
                va='center', fontsize=7.5, color='#444')
    leyenda = [mpatches.Patch(color=c, label=t)
        for t, c in color_tipo.items() if t in df_top['Tipo'].values]
    ax.legend(handles=leyenda, fontsize=8, loc='lower right')
    ax.axvline(0, color=COLORS['neutral'], linewidth=0.8, alpha=0.5)
    ax.set_xlabel(f'Reducción en {metrica} (permutation importance)', fontsize=9)
    ax.set_title(titulo, fontsize=10, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)

plt.suptitle('Importancia de variables — Comparativa DCSA vs DSSA',
    fontsize=13, fontweight='bold')
plt.tight_layout()
fig.savefig(OUT_FIG / 'reporte_importancia_comparativa.png',
    dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

## 4. Validación temporal — Robustez de los modelos

### DCSA — Ruptura estructural 2021–2022

| Escenario | F1 | Bal. Acc | Specificity | Brier |
|-----------|----|---------|-----------|---------|
| A. Test aleatorio (03a) | 0.800 | 0.737 | 0.699 | 0.182 |
| B. Modelo 03a → test temporal 2022-2025 | 0.933 | 0.564 | 0.143 | 0.211 |
| C. Re-entrenado temporalmente | 0.862 | 0.569 | 0.165 | **0.171** |

El F1=0.933 del escenario B es un artefacto: con 87% de hospitalizados
en 2022–2025, predecir siempre 'hospitalizar' da ese resultado.
La Balanced Accuracy (0.564 ≈ azar) revela la realidad.
El Brier Score del modelo re-entrenado (0.171) es el mejor de los tres —
mejor calibración aunque discriminación similar.

**Causa raíz:** el drift de FIEBRE de 21.4% a 100% es imposible clínicamente.
El cambio es en el protocolo de llenado del formulario, no en los pacientes.

### DSSA — Sin ruptura estructural

La tasa de hospitalización DSSA es estable entre períodos (~3.7% → ~4.0%).
El drift es epidemiológico real — ERUPCIONR cae significativamente,
coherente con cambios en serotipos circulantes.
Los cambios en métricas entre escenarios reflejan degradación real,
no artefactos distribucionales como en DCSA.

**Contraste fundamental:** DCSA tiene ruptura metodológica (protocolo INS),
DSSA tiene drift epidemiológico real — dos tipos de inestabilidad
con implicaciones distintas para el re-entrenamiento.

In [ ]:
# Cargar tabla comparativa temporal si existe
tab_temp_dcsa = OUT_TAB / 'comparativa_temporal_DCSA.csv'
tab_temp_dssa = OUT_TAB / 'comparativa_temporal_DSSA.csv'

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, path, titulo, color_escenarios in zip(
    axes,
    [tab_temp_dcsa, tab_temp_dssa],
    ['DCSA — Validación temporal', 'DSSA — Validación temporal'],
    [
        [COLORS['neutral'], COLORS['secondary'], COLORS['primary']],
        [COLORS['neutral'], COLORS['secondary'], COLORS['primary']],
    ]
):
    if not path.exists():
        ax.text(0.5, 0.5, 'Tabla no disponible\nEjecutar 03b/03d primero',
            ha='center', va='center', transform=ax.transAxes, fontsize=10)
        ax.set_title(titulo)
        continue

    df_t = pd.read_csv(path)
    metricas = [c for c in ['F1','AUC-ROC','Bal_Acc','Recall','Specificity']
        if c in df_t.columns]
    x = np.arange(len(metricas))
    w = 0.26
    for i, (_, row) in enumerate(df_t.iterrows()):
        vals = [row.get(m, 0) for m in metricas]
        label = row.get('Escenario', f'Escenario {i}').replace('\n',' ')
        ax.bar(x + (i-1)*w, vals, w,
            color=color_escenarios[i % len(color_escenarios)],
            alpha=0.85, label=label[:30], edgecolor='white')
    ax.axhline(0.5, color='red', linestyle=':', linewidth=1.2, alpha=0.5,
        label='Bal. Acc = 0.5 (azar)')
    ax.set_xticks(x)
    ax.set_xticklabels(metricas, fontsize=9)
    ax.set_ylim(0, 1.15)
    ax.set_title(titulo)
    ax.legend(fontsize=7, loc='upper right')
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Validación temporal — Tres escenarios por estrato',
    fontsize=13, fontweight='bold')
plt.tight_layout()
fig.savefig(OUT_FIG / 'reporte_validacion_temporal.png',
    dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

## 5. Hallazgos del clustering

El análisis no supervisado reveló que la estructura de los datos
está condicionada por la calidad del registro, no solo por los
perfiles clínicos reales.

### DCSA — Cluster de formularios incompletos

| Cluster | N | Hosp. | Perfil |
|---------|---|-------|--------|
| 0 | 4,717 | 38.0% | **FIEBRE=0, todos los síntomas=0** — formularios vacíos |
| 1 | 126 | 84.1% | Alta carga de signos: DOLOR_ABDO=0.635, alta hospitalización |
| 2 | 7,870 | 80.6% | Alta prevalencia síntomas generales — perfil clínico completo |

El Cluster 0 (4,717 casos, 38% hosp.) **no representa pacientes sin síntomas**
— representa casos clasificados como DCSA con formularios incompletos
del período 2015–2021. Este cluster es la evidencia más directa del
sesgo de llenado documentado en el 03b.

**ARI K-Means vs K-Prototypes = 0.004:** los dos métodos encuentran
estructuras completamente distintas — la segmentación depende fuertemente
del método, señal de estructura débil o dependiente de supuestos.

**UMAP + HDBSCAN:** identificó 4 clusters con 13.1% de outliers —
casos con presentaciones clínicas inusuales que merecen atención especial.

### DSSA — Clusters dominados por desbalance

| Cluster | N | Hosp. |
|---------|---|-------|
| 0 | 2,396 | 5.1% |
| 1 | 13,337 | 3.7% |
| 2 | 1 | 100.0% — caso singular |

K-Means DSSA con k=2 tiene Silhouette=0.975 — artificialmente alto
porque separa un único cluster denso del resto del espacio.
Los clusters con k=3 tienen tasas de hospitalización muy similares
(3.7%–5.1%), sin significado clínico diferenciado.

In [ ]:
# Figura resumen de clustering: perfiles por cluster
try:
    tab_dcsa_cl = pd.read_csv(OUT_TAB / 'clustering_dcsa_perfiles.csv', index_col=0)
    tab_dssa_cl = pd.read_csv(OUT_TAB / 'clustering_dssa_perfiles.csv', index_col=0)
    tiene_clustering = True
except FileNotFoundError:
    tiene_clustering = False
    print('Tablas de clustering no disponibles — ejecutar 04 primero')

if tiene_clustering:
    SIGNOS   = ['DOLOR_ABDO','VOMITO','DIARREA','SOMNOLENCI','HIPOTENSIO',
                'HEPATOMEG','HEM_MUCOSA','HIPOTERMIA','AUM_HEMATO','CAIDA_PLAQ','ACUM_LIQUI']
    SINTOMAS = ['FIEBRE','CEFALEA','DOLRRETROO','MALGIAS','ARTRALGIA','ERUPCIONR']
    vars_cl  = [v for v in SINTOMAS + SIGNOS if v in tab_dcsa_cl.columns]

    fig, axes = plt.subplots(1, 2, figsize=(18, 4))
    for ax, tabla, titulo in zip(
        axes,
        [tab_dcsa_cl[vars_cl], tab_dssa_cl[[v for v in vars_cl if v in tab_dssa_cl.columns]]],
        ['DCSA — Perfil clínico por cluster', 'DSSA — Perfil clínico por cluster']
    ):
        im = ax.imshow(tabla.values, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)
        plt.colorbar(im, ax=ax, label='Prevalencia media', shrink=0.8)
        ax.set_xticks(range(len(tabla.columns)))
        ax.set_xticklabels(tabla.columns, rotation=45, ha='right', fontsize=7)
        hosp = tabla.index.tolist()
        ax.set_yticks(range(len(tabla)))
        ax.set_yticklabels([f'Cluster {k}' for k in tabla.index], fontsize=9)
        ax.set_title(titulo, fontsize=10, fontweight='bold')
        for i in range(len(tabla)):
            for j, v in enumerate(tabla.values[i]):
                ax.text(j, i, f'{v:.2f}', ha='center', va='center',
                    fontsize=6.5, color='black' if 0.2<v<0.8 else 'white')

    plt.suptitle('Heatmap de perfiles clínicos por cluster — DCSA y DSSA',
        fontsize=12, fontweight='bold')
    plt.tight_layout()
    fig.savefig(OUT_FIG / 'reporte_clustering_heatmaps.png',
        dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()

## 6. Implicaciones clínicas

### Para el sistema de salud de Bucaramanga

**DCSA — modo Screening-First:**
En epidemias con alta mortalidad o baja capacidad de seguimiento ambulatorio,
ajustar el umbral a 0.243 garantiza Recall=95.2% — se pierden menos del 5%
de los hospitalizados reales. El costo es hospitalizar al 89% de la cohorte
— operacionalmente exigente pero clínicamente seguro.

**DCSA — modo Capacity-First:**
Cuando las camas de observación están al límite, el Top 5% (umbral=0.846)
identifica casos con PPV=97.7% — casi todos los que alerta realmente
necesitan ingreso. El enriquecimiento (1.51x) es modesto pero real.

**DSSA — Capacity-First obligatorio:**
Con 3.9% de prevalencia, el umbral 0.5 es inapropiado.
El Top 5–10% de mayor riesgo ofrece el mejor equilibrio entre
cobertura y viabilidad operacional. Un sistema de alerta basado en
este modelo debe definir explícitamente el umbral operacional.

### Para el sistema de vigilancia epidemiológica (INS/SIVIGILA)

1. **Estandarizar el llenado del formulario:** el drift de FIEBRE de 21% a 100%
   entre 2015–2021 y 2022–2025 invalida los modelos históricos y contamina
   el clustering. Un formulario con campos obligatorios evitaría este problema.
2. **Incluir variables de laboratorio:** hemograma y hematocrito son predictores
   clave que el SIVIGILA no captura — su inclusión probablemente rompería
   el techo informativo observado en ambos estratos.
3. **Monitoreo continuo de drift:** el cambio de protocolo 2021 degradó
   silenciosamente los modelos sin alarma. Un sistema de monitoreo de
   Balanced Accuracy y Brier Score en producción detectaría estos cambios.

## 7. Limitaciones y trabajo futuro

### Limitaciones

| Limitación | Impacto | Estrato afectado |
|-----------|---------|------------------|
| Sesgo de llenado pre-2021 | Alto — invalida modelos históricos | DCSA |
| Ausencia de laboratorio en SIVIGILA | Alto — techo informativo | Ambos |
| Desbalance severo 96/4 | Moderado — SMOTE mitiga | DSSA |
| Sin validación geográfica externa | Moderado — generalización incierta | Ambos |
| Variables demográficas en DSSA | Alerta equidad — codifica inequidades | DSSA |
| Clustering captura sesgo de registro | Moderado — limita interpretación | DCSA |

### Trabajo futuro

1. **Enriquecer los features:** integrar resultados de laboratorio
   (hemograma, hematocrito) disponibles en los registros clínicos
   de las IPS — el paso más impactante para mejorar el rendimiento.
2. **Modelo exclusivo post-2021 para DCSA:** con 87% de hospitalización
   en 2022–2025, replantear el problema como detección de ambulatorios
   (¿quién NO necesita hospitalización?) en lugar de hospitalizados.
3. **Validación externa:** aplicar el modelo a datos de Floridablanca,
   Girón y Piedecuesta — municipios del área metropolitana con acceso
   al mismo sistema de vigilancia.
4. **Auditoría de equidad:** evaluar si la importancia de variables
   demográficas en DSSA produce recomendaciones diferentes para
   pacientes con mismos síntomas pero distinto estrato socioeconómico.
5. **Re-entrenamiento continuo:** implementar pipeline de monitoreo
   mensual con alertas cuando Balanced Accuracy o Brier Score
   degraden más de 5 puntos porcentuales respecto al baseline.

In [ ]:
# Figura resumen final — panel comparativo DCSA vs DSSA
fig = plt.figure(figsize=(18, 10))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

# Panel 1: prevalencia comparativa
ax1 = fig.add_subplot(gs[0, 0])
estratos = ['DCSA\n(con signos)', 'DSSA\n(sin signos)']
hosp     = [64.8, 3.9]
colores  = [COLORS['con_signos'], COLORS['sin_signos']]
bars = ax1.bar(estratos, hosp, color=colores, alpha=0.85, edgecolor='white', width=0.5)
for bar, val in zip(bars, hosp):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
        f'{val}%', ha='center', fontsize=11, fontweight='bold')
ax1.set_ylabel('% hospitalizados')
ax1.set_title('Prevalencia de hospitalización')
ax1.set_ylim(0, 80)
ax1.grid(axis='y', alpha=0.3)

# Panel 2: métricas clave en test
ax2 = fig.add_subplot(gs[0, 1])
metricas_k = ['F1 / AUC-PR', 'Bal. Acc.', 'Recall', 'Brier']
# Obtener valores robustamente
def get_val(tabla, metrica, default=0):
    for col in [metrica, metrica.replace('-','_'), metrica.replace(' ','_')]:
        if col in tabla.columns:
            return tabla.iloc[0][col]
    return default

vals_dcsa = [
    get_val(tabla_dcsa, 'F1'),
    get_val(tabla_dcsa, 'Bal. Acc', get_val(tabla_dcsa, 'Bal_Acc')),
    get_val(tabla_dcsa, 'Recall'),
    get_val(tabla_dcsa, 'Brier Score', get_val(tabla_dcsa, 'Brier')),
]
vals_dssa = [
    get_val(tabla_dssa, 'AUC-PR', get_val(tabla_dssa, 'AUC_PR')),
    get_val(tabla_dssa, 'Bal. Acc', get_val(tabla_dssa, 'Bal_Acc')),
    get_val(tabla_dssa, 'Recall'),
    get_val(tabla_dssa, 'Brier Score', get_val(tabla_dssa, 'Brier')),
]
x = np.arange(len(metricas_k))
w = 0.35
ax2.bar(x-w/2, vals_dcsa, w, color=COLORS['con_signos'], alpha=0.85, label='DCSA', edgecolor='white')
ax2.bar(x+w/2, vals_dssa, w, color=COLORS['sin_signos'], alpha=0.85, label='DSSA', edgecolor='white')
ax2.set_xticks(x)
ax2.set_xticklabels(metricas_k, fontsize=8)
ax2.set_ylim(0, 1.1)
ax2.legend(fontsize=9)
ax2.set_title('Métricas clave en test\n(Brier: menor = mejor)')
ax2.grid(axis='y', alpha=0.3)

# Panel 3: Balanced Accuracy validación temporal
ax3 = fig.add_subplot(gs[0, 2])
escenarios = ['A\nTest aleatorio', 'B\nModelo→temporal', 'C\nRe-entrenado']
bal_dcsa   = [0.737, 0.564, 0.569]
bal_dssa   = [None, None, None]  # se completará si la tabla existe
if tab_temp_dssa.exists():
    df_td = pd.read_csv(tab_temp_dssa)
    if 'Bal_Acc' in df_td.columns:
        bal_dssa = df_td['Bal_Acc'].tolist()[:3]

x3 = np.arange(len(escenarios))
ax3.plot(x3, bal_dcsa, color=COLORS['con_signos'], linewidth=2,
    marker='o', markersize=8, label='DCSA')
if any(v is not None for v in bal_dssa):
    ax3.plot(x3, [v or 0 for v in bal_dssa], color=COLORS['sin_signos'],
        linewidth=2, marker='s', markersize=8, label='DSSA')
ax3.axhline(0.5, color='red', linestyle=':', linewidth=1.5,
    alpha=0.7, label='Referencia azar')
ax3.set_xticks(x3)
ax3.set_xticklabels(escenarios, fontsize=8)
ax3.set_ylim(0.4, 0.85)
ax3.set_ylabel('Balanced Accuracy')
ax3.set_title('Robustez temporal\nBalanced Accuracy por escenario')
ax3.legend(fontsize=9)
ax3.grid(alpha=0.3)

# Panel 4: importancia promedio por tipo de variable
ax4 = fig.add_subplot(gs[1, 0:2])
tipos = ['Signo de alarma', 'Sintoma general', 'Demografica', 'Estacionalidad']
colores_t = [COLORS['con_signos'], COLORS['sin_signos'], COLORS['accent'], COLORS['neutral']]

for estrato, df_imp, color_e, offset in [
    ('DCSA', imp_dcsa, COLORS['con_signos'], -0.2),
    ('DSSA', imp_dssa, COLORS['sin_signos'],  0.2)
]:
    df_imp_f = df_imp[df_imp['Variable'] != 'AÑO'].copy()
    medias = df_imp_f.groupby('Tipo')['Importancia'].mean().reindex(tipos, fill_value=0)
    ax4.bar(
        np.arange(len(tipos)) + offset, medias.values, 0.35,
        color=color_e, alpha=0.85, label=estrato, edgecolor='white'
    )

ax4.set_xticks(np.arange(len(tipos)))
ax4.set_xticklabels(tipos, fontsize=9)
ax4.set_ylabel('Importancia media')
ax4.set_title('Importancia promedio por tipo de variable\nDCSA vs DSSA')
ax4.legend(fontsize=9)
ax4.grid(axis='y', alpha=0.3)

# Panel 5: ARI clustering
ax5 = fig.add_subplot(gs[1, 2])
metodos = ['K-Means\nvs\nK-Prototypes', 'K-Means\nvs\nGMM']
ari_vals = [0.004, 0.748]  # valores reales del 04
colores_ari = [COLORS['secondary'], COLORS['verde']]
bars5 = ax5.bar(metodos, ari_vals, color=colores_ari, alpha=0.85,
    edgecolor='white', width=0.5)
for bar, val in zip(bars5, ari_vals):
    ax5.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02,
        f'ARI={val:.3f}', ha='center', fontsize=10, fontweight='bold')
ax5.axhline(0.5, color=COLORS['neutral'], linestyle='--', linewidth=1.2,
    label='Alta concordancia (0.5)')
ax5.set_ylim(0, 1.1)
ax5.set_ylabel('Adjusted Rand Index')
ax5.set_title('Concordancia entre métodos\nde clustering — DCSA')
ax5.legend(fontsize=8)
ax5.grid(axis='y', alpha=0.3)

plt.suptitle(
    'Panel de Resultados — Predicción de Hospitalización por Dengue\n'
    'Bucaramanga 2015–2025 | SIVIGILA',
    fontsize=14, fontweight='bold', y=1.01)
plt.savefig(OUT_FIG / 'reporte_panel_final.png',
    dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print('Panel final guardado en outputs/figures/reporte_panel_final.png')

---

## Cierre

Este estudio demuestra que el análisis de datos de vigilancia epidemiológica
de rutina puede aportar información predictiva clínicamente útil,
pero también expone las limitaciones estructurales de estos sistemas:
cambios de protocolo no documentados, formularios incompletos y ausencia
de variables clínicas clave pueden invalidar modelos aparentemente robustos.

La contribución más importante no es el F1=0.800 del DCSA sino el conjunto
de hallazgos metodológicos: la necesidad de reportar métricas múltiples
(F1 + Balanced Accuracy + Brier Score), la validación temporal como
requisito antes del despliegue, y la interpretación cuidadosa de los
resultados del clustering cuando los datos tienen sesgos de calidad.

> *'Un modelo predictivo desplegado sin monitoreo continuo es un modelo
> que deja de ser válido en silencio.'*